# Course Generation Model: Xây dựng chương trình học tập cá nhân hóa

**Mục tiêu**: Xây dựng hệ thống gợi ý môn học dựa trên ràng buộc chương trình đào tạo, prerequisites, và semantic weights.

**Tác giả**: IT2041.CH201 Course Generation System

---

## 📌 Tổng quan

Hệ thống gồm 2 phần chính:
1. **CourseGenerationModel**: Xử lý ràng buộc chương trình đào tạo (program constraints) + prerequisites
2. **GTECourseWeightedGenerator**: Kết hợp semantic weights từ GTE để xếp hạng môn học

**Dữ liệu đầu vào**:
- `data/raw/course_catalog.csv`: Danh mục môn học
- `data/raw/course_descriptions.json`: Mô tả chi tiết từ daa.uit.edu.vn (tùy chọn)
- `data/rules/local/`: Ràng buộc chương trình đào tạo theo từng ngành
- `data/rules/global/course_prerequisites_catalog.json`: Prerequisites toàn cục
- `notebooks/anchor_profiles.json`: Anchor profiles cho 5 Khoa

## 📦 Cài đặt thư viện

In [ ]:
!pip install sentence-transformers scikit-learn pandas numpy

In [ ]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

print("✅ Import thành công")

## 📂 Load dữ liệu

In [ ]:
ROOT = Path("..").resolve()

# Load course catalog
df_catalog = pd.read_csv(ROOT / "data" / "raw" / "course_catalog.csv")

# Merge với descriptions từ daa.uit.edu.vn (nếu có)
desc_file = ROOT / "data" / "raw" / "course_descriptions.json"
if desc_file.exists():
    with open(desc_file, 'r', encoding='utf-8') as f:
        descriptions = json.load(f)
    df_desc = pd.DataFrame(descriptions)
    # Merge theo course_id
    df_catalog = df_catalog.merge(df_desc[['course_id', 'description']], on='course_id', how='left')
    print(f"✅ Loaded {len(descriptions)} course descriptions from daa.uit.edu.vn")
else:
    df_catalog['description'] = ''
    print(f"⚠️  No course_descriptions.json found. Sẽ dùng tên môn thay vì mô tả chi tiết.")

print(f"✅ Catalog: {len(df_catalog)} courses")
print(f"\nMẫu dữ liệu:")
print(df_catalog[['course_id', 'name_vi', 'managed_by', 'course_type', 'description']].head(10))

## 🏗 Class 1: CourseGenerationModel

**Công việc**: Xử lý ràng buộc chương trình đào tạo và prerequisites.

**Chức năng**:
- `get_course_credits()`: Lấy số tín chỉ của môn
- `get_prerequisites()`: Lấy danh sách môn tiên quyết
- `can_take_course()`: Kiểm tra sinh viên có thể học môn không
- `get_eligible_courses()`: Lấy danh sách môn đủ điều kiện học

In [ ]:
class CourseGenerationModel:
    """
    Mô hình sinh môn học dựa trên ràng buộc.
    Dùng để get eligible courses + prerequisites.
    """
    def __init__(self, constraints: Dict, global_prerequisites: Dict, catalog_df: pd.DataFrame):
        self.constraints = constraints
        self.global_prerequisites = global_prerequisites
        self.catalog_df = catalog_df
        self.all_courses = set(constraints.get('course_blocks', {}).keys())
    
    def get_course_credits(self, course_id: str) -> int:
        row = self.catalog_df[self.catalog_df['course_id'] == course_id]
        if row.empty:
            return 0
        return int(row.iloc[0].get('credits_lt', 0)) + int(row.iloc[0].get('credits_th', 0))
    
    def get_prerequisites(self, course_id: str) -> List[str]:
        return self.global_prerequisites.get(course_id, [])
    
    def can_take_course(self, course_id: str, completed_courses: Set[str]) -> bool:
        prerequisites = self.get_prerequisites(course_id)
        return all(prereq in completed_courses for prereq in prerequisites)
    
    def get_eligible_courses(self, completed_courses: Set[str]) -> List[str]:
        eligible = []
        for course_id in self.all_courses:
            if course_id not in completed_courses and self.can_take_course(course_id, completed_courses):
                eligible.append(course_id)
        return eligible

## 📊 Load Constraints và Prerequisites

In [ ]:
def load_course_constraints(cohort: str, major: str) -> Dict:
    rules_file = ROOT / "data" / "rules" / "local" / cohort / f"{major}.json"
    if not rules_file.exists():
        print(f"⚠️  Rules file not found: {rules_file}")
        return {}
    with open(rules_file, 'r', encoding='utf-8') as f:
        rules_data = json.load(f)
    constraints = {'program_structure': None, 'course_groups': [],
                   'prerequisites': {}, 'course_blocks': {}, 'total_credits': 0}
    for rule in rules_data.get('rules', []):
        rule_type = rule.get('rule_type')
        if rule_type == 'PROGRAM_STRUCTURE':
            constraints['program_structure'] = rule
            constraints['total_credits'] = rule.get('total_credits', 0)
            for block in rule.get('knowledge_blocks', []):
                block_id = block.get('block_id')
                for course_id in block.get('courses', []):
                    constraints['course_blocks'][course_id] = block_id
        elif rule_type == 'COURSE_GROUP_REQUIREMENT':
            constraints['course_groups'].append(rule)
    return constraints

def load_global_prerequisites() -> Dict[str, List[str]]:
    prereq_file = ROOT / "data" / "rules" / "global" / "course_prerequisites_catalog.json"
    if not prereq_file.exists():
        return {}
    with open(prereq_file, 'r', encoding='utf-8') as f:
        prereq_data = json.load(f)
    prerequisites = {}
    for rule in prereq_data.get('rules', []):
        if rule.get('rule_type') == 'COURSE_PREREQUISITE':
            payload = rule.get('payload', {})
            course = payload.get('course', {})
            course_id = course.get('course_id')
            requires_all = payload.get('requires_all', [])
            prereq_ids = [c.get('course_id') for c in requires_all if c.get('course_id')]
            if course_id and prereq_ids:
                prerequisites[course_id] = prereq_ids
    return prerequisites

# Load cho KTPM K2023
constraints = load_course_constraints('K2023', 'KTPM')
global_prerequisites = load_global_prerequisites()
print(f"✅ Constraints: {len(constraints.get('course_blocks', {}))} courses")
print(f"✅ Prerequisites: {len(global_prerequisites)} courses")

# Giả lập môn đã học
completed_courses = {
    'IT001', 'IT002', 'IT003', 'IT004', 'IT005',
    'MA003', 'MA004', 'MA005', 'MA006',
    'SS003', 'SS007', 'SS008', 'SS009', 'SS010', 'SS006',
    'ENG01', 'ENG02', 'ENG03',
    'ME001', 'PE231', 'PE232'
}
print(f"✅ Giả lập {len(completed_courses)} môn đã học")

## 🚀 Demo: Gợi ý môn học kỳ tới

In [ ]:
# Khởi tạo model
base_model = CourseGenerationModel(constraints, global_prerequisites, df_catalog)

# Lấy môn đủ điều kiện
eligible = base_model.get_eligible_courses(completed_courses)
print(f"\n📊 {len(eligible)} môn đủ điều kiện học kỳ tới:")

# Sắp xếp theo số tín chỉ
eligible_with_credits = [(cid, base_model.get_course_credits(cid)) for cid in eligible]
eligible_with_credits.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 20 môn có nhiều tín chỉ nhất:")
for cid, credits in eligible_with_credits[:20]:
    row = df_catalog[df_catalog['course_id'] == cid]
    name = row.iloc[0]['name_vi'] if not row.empty else ''
    prereqs = base_model.get_prerequisites(cid)
    prereq_str = ', '.join(prereqs) if prereqs else 'Không có'
    print(f"  {cid:10s} | {name[:35]:35s} | {credits:2d} TC | Prereq: {prereq_str[:30]}")

## 🔗 Tích hợp với GTECourseWeightedGenerator

Để sử dụng semantic weights, kết hợp với `GTECourseWeightedGenerator` từ notebook `zero_shot_semantic_matching.ipynb`.

```python
from notebooks.zero_shot_semantic_matching import GTESemanticMatcher, GTECourseWeightedGenerator

# Khởi tạo semantic matcher
semantic_matcher = GTESemanticMatcher()
semantic_matcher.fit(df_catalog)

# Khởi tạo weighted generator
weighted_gen = GTECourseWeightedGenerator(
    semantic_matcher=semantic_matcher,
    constraints=constraints,
    global_prerequisites=global_prerequisites,
    catalog_df=df_catalog
)

# Gợi ý môn học cho định hướng cụ thể
result = weighted_gen.recommend_weighted_courses(
    completed_courses=completed_courses,
    max_credits=25,
    dept_code='KHMT',
    target_anchor='ai_ml'
)
```